<a href="https://colab.research.google.com/github/CibrianA04/AnalisisDATASETS/blob/main/Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

La RAM (Memory de Acceso Aleatorio, también conocida como memoria RAM) y el almacenamiento en una computadora son componentes diferentes que desempeñan funciones importantes para el funcionamiento de la máquina.

**RAM (Memoria RAM):**

La RAM es un tipo de memoria temporal que almacena datos y programas que están siendo ejecutados actualmente por el procesador. Se trata de una memoria volátil, lo que significa que los datos se pierden cuando se apaga la computadora. La RAM es como un tablero de trabajo en donde se colocan las herramientas y los materiales que se están utilizando en el momento.

La función principal de la RAM es almacenar la siguiente información:

- Datos del programa que está ejecutándose
- Datos del usuario que están siendo procesados
- Estado de las aplicaciones en ejecución

**Almacenamiento:**

El almacenamiento en una computadora (también conocido como almacenamiento de disco) es un tipo de memoria que almacena datos y programas de manera permanente para su uso 

In [ ]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "id": "chatcmpl-014d5a86-cf2a-44d9-be73-8396deb4cf34",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La RAM (Memory de Acceso Aleatorio, también conocida como memoria RAM) y el almacenamiento en una computadora son componentes diferentes que desempeñan funciones importantes para el funcionamiento de la máquina.\n\n**RAM (Memoria RAM):**\n\nLa RAM es un tipo de memoria temporal que almacena datos y programas que están siendo ejecutados actualmente por el procesador. Se trata de una memoria volátil, lo que significa que los datos se pierden cuando se apaga la computadora. La RAM es como un tablero de trabajo en donde se colocan las herramientas y los materiales que se están utilizando en el momento.\n\nLa función principal de la RAM es almacenar la siguiente información:\n\n- Datos del programa que está ejecutándose\n- Datos del usuario que están siendo procesados\n- Estado de las aplicaciones en e

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 53
Tokens de la respuesta: 554
Tokens totales: 607


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.00 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.00 s — 607 tokens
Modelo grande: 2.12 s — 661 tokens

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) y el almacenamiento son dos componentes fundamentales de una computadora, pero cumplen funciones distintas y tienen características diferentes.

**RAM (Memoria de Acceso Aleatorio)**

La RAM es un tipo de memoria volátil, lo que significa que su contenido se pierde cuando la computadora se apaga. La RAM se utiliza para almacenar temporalmente los datos y los programas que se están utilizando en ese momento. Cuando una aplicación se ejecuta, se carga en la RAM para que el procesador pueda acceder a ella rápidamente.

La RAM tiene las siguientes características:

*   **Velocidad:** La RAM es muy rápida, ya que los datos se pueden leer y escritura en ella en cuestión de nanosegundos.
*   **Capacidad:** La cantidad de RAM disponible en una computadora es limitada, generalmente entre 4 y 64 GB, dependiendo del modelo y la cantidad de ranuras disponibles.
